In [1]:
# 07c_governance_honest_eval.ipynb
# =============================================================================
# Notebook 07c — Honest Governance Evaluation
#
# Fixes the two invalid evaluations:
#   - nb07  : hard-clipped CFs -> compliance 100% by construction (tautology)
#   - nb07b : permitted_range only -> violations conflate optimiser failure
#             with guardrail-quality failure
#
# Principles (see EVAL_DESIGN_07c.md):
#   1. Scoring criteria are EXTERNAL and FIXED (WHO / KDRI absolute values),
#      independent of the Hard-Rule generation formulas -> no tautology.
#   2. Report BOTH soft_compliance (CF as generated, no clip) and
#      hard_compliance (after deterministic hard-rule clip). The gap is the
#      MEASURED contribution of the deterministic layer.
#   3. Attribute every violation: range_respected (optimiser) vs
#      range_permits_violation (guardrail design).
#
# Conditions: PreGuardrail / LLM_only / HardRule_only / LLM+Hard(soft) / LLM+Hard(hard)
#
# Input  : ../results/tables/agent_config.pkl, df_final.pkl, model_{group}.pkl,
#          ../results/tables/guardrail_ranges_v2.json
# Output : ../results/tables/governance_07c_percf.csv
#          ../results/tables/governance_07c_summary.csv
#          ../results/tables/governance_07c_mcnemar.csv
#          ../results/tables/governance_07c_attribution.csv
# =============================================================================

# %%
# ## 0. Imports
import json, joblib, warnings
import numpy as np, pandas as pd
import dice_ml
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings('ignore')

# %%
# ## 1. External fixed scoring thresholds (WHO / KDRI)
# IMPORTANT: these are EXTERNAL absolute values, NOT ratio-of-baseline formulas.
# They must be confirmed with the corresponding author / a dietitian before the
# submitted revision. See EVAL_DESIGN_07c.md for sources.

EXT = {
    'energy_floor_kcal':   800.0,    # clinical VLCD lower boundary
    'protein_g_per_kg':    0.83,     # KDRI 2020 RNI basis (uses CF Weight)
    'potassium_floor_mg':  2000.0,   # conservative deficiency guard (< AI 3500)
    'fiber_g_per_1000kcal': 12.0,    # dietary-fibre AI density
    'carb_floor_g':        100.0,    # CNS glucose minimum / anti-ketosis
    'sodium_who_mg':       2000.0,   # WHO adult recommendation (context only)
    'change_tol_pct':      1.0,      # treat |change| <= 1% as "no change"
}

# %%
# ## 2. Load data
agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')
X_FEATURES      = agent_config['X_features']
VARY_FEATURES   = agent_config['vary_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']

models = {g: joblib.load(f'../results/tables/model_{g}.pkl') for g in AGEGROUP_CONFIG}

with open('../results/tables/guardrail_ranges_v2.json', encoding='utf-8') as f:
    GR = json.load(f)

# %%
# ## 3. EXTERNAL scoring functions (independent of range-generation formulas)

def g1_physical_consistency(cf, orig, tol=EXT['change_tol_pct']):
    """BMI, WaistCirc, Weight move in the same direction (unchanged concept)."""
    def sign(k):
        o = float(orig.get(k, 0)); v = float(cf.get(k, o))
        pct = (v - o) / o * 100 if o != 0 else 0
        return np.sign(pct) if abs(pct) > tol else 0
    s = [sign(k) for k in ['BMI', 'WaistCirc', 'Weight']]
    nz = [x for x in s if x != 0]
    return True if not nz else len(set(nz)) == 1

def g2_energy_safety_ext(cf, orig):
    """External: CF energy >= 800 kcal absolute AND <= baseline (recourse dir)."""
    c = float(orig.get('Energy_kcal', 0)); v = float(cf.get('Energy_kcal', c))
    return (v >= EXT['energy_floor_kcal']) and (v <= c * 1.01)

def g3_conflict_prevention(cf, orig, tol=EXT['change_tol_pct']):
    """If sodium decreases, carb and sugar must not increase (unchanged concept)."""
    c_na = float(orig.get('Sodium_mg', 0)); v_na = float(cf.get('Sodium_mg', c_na))
    if not (c_na > 0 and (c_na - v_na) / c_na * 100 > tol):
        return True
    c_c = float(orig.get('Carb_g', 0));  v_c = float(cf.get('Carb_g', c_c))
    c_s = float(orig.get('Sugar_g', 0)); v_s = float(cf.get('Sugar_g', c_s))
    carb_ok  = (v_c - c_c) / c_c * 100 <= tol if c_c > 0 else True
    sugar_ok = (v_s - c_s) / c_s * 100 <= tol if c_s > 0 else True
    return carb_ok and sugar_ok

def g4_macronutrient_floors_ext(cf, orig):
    """External absolute floors (KDRI-based), NOT ratio-of-baseline.
       Protein and fibre are scaled to CF body weight / CF energy respectively,
       so the floor does not depend on the patient's own baseline intake."""
    weight = float(cf.get('Weight', orig.get('Weight', 60.0)))
    energy = float(cf.get('Energy_kcal', orig.get('Energy_kcal', 1500.0)))
    protein_floor = EXT['protein_g_per_kg'] * weight
    fiber_floor   = EXT['fiber_g_per_1000kcal'] * (energy / 1000.0)
    checks = [
        ('Protein_g',    protein_floor),
        ('Potassium_mg', EXT['potassium_floor_mg']),
        ('Carb_g',       EXT['carb_floor_g']),
        ('Fiber_g',      fiber_floor),
    ]
    for feat, floor in checks:
        v = float(cf.get(feat, orig.get(feat, 0)))
        if v < floor * 0.99:   # 1% tolerance
            return False
    return True

def score_all(cf, orig):
    return {
        'G1': int(g1_physical_consistency(cf, orig)),
        'G2': int(g2_energy_safety_ext(cf, orig)),
        'G3': int(g3_conflict_prevention(cf, orig)),
        'G4': int(g4_macronutrient_floors_ext(cf, orig)),
    }

# %%
# ## 4. Range utilities

def build_permitted(ranges, df_ref, features, only_these=None):
    """permitted_range for DiCE. LLM_only omits features the LLM didn't supply."""
    sr = {}
    feats = features if only_these is None else [f for f in features if f in only_these]
    for feat in feats:
        pair = ranges.get(feat)
        if pair is None:
            continue
        lo, hi = float(pair[0]), float(pair[1])
        data_lo = float(df_ref[feat].quantile(0.05))
        data_hi = float(df_ref[feat].quantile(0.95))
        sr[feat] = [round(max(min(lo, data_lo), 0.0), 4), round(max(hi, data_hi), 4)]
    return sr

def clip_to_ranges(cf, ranges):
    out = dict(cf)
    for feat, pair in ranges.items():
        if feat in out and pair is not None:
            out[feat] = float(np.clip(float(out[feat]), float(pair[0]), float(pair[1])))
    return out

def range_respected(cf, permitted):
    """Did the generated CF stay within the injected permitted_range?"""
    for feat, (lo, hi) in permitted.items():
        if feat in cf:
            v = float(cf[feat])
            if v < lo - 1e-6 or v > hi + 1e-6:
                return False
    return True

def range_permits_violation(orig, ranges):
    """Does the injected range ITSELF permit an external violation?
       Checks whether the range's worst admissible corner can fail G2/G4."""
    # Construct the 'worst' admissible point for the external floors:
    worst = dict(orig)
    for feat in ['Energy_kcal', 'Protein_g', 'Potassium_mg', 'Carb_g', 'Fiber_g', 'Weight']:
        if feat in ranges and ranges[feat] is not None:
            worst[feat] = float(ranges[feat][0])   # lower bound = worst for floors
    g2 = g2_energy_safety_ext(worst, orig)
    g4 = g4_macronutrient_floors_ext(worst, orig)
    return int(not (g2 and g4))

def gen_cfs(exp, query, permitted=None, n=4):
    kw = dict(total_CFs=n, desired_class=0, features_to_vary=VARY_FEATURES,
              proximity_weight=0.2, sparsity_weight=0.1)
    if permitted:
        kw['permitted_range'] = permitted
    try:
        cf = exp.generate_counterfactuals(query, **kw)
        return cf.cf_examples_list[0].final_cfs_df.to_dict('records') if cf else []
    except Exception:
        return []

# %%
# ## 5. Evaluate all conditions
records = []
attribution = []

for case_key, g in GR.items():
    grp = g['group']; mdl = models.get(grp)
    if mdl is None:
        continue
    orig = g['patient_profile']
    cfg  = AGEGROUP_CONFIG[grp]
    age  = 0.0 if cfg['age_min'] < 60 else 1.0
    dref = df_final[(df_final['AgeGroup'] == age) & (df_final['Sex'] == cfg['sex_code'])].copy()
    query = pd.DataFrame([orig])[X_FEATURES]

    d = dice_ml.Data(dataframe=df_final.copy().astype(float)[X_FEATURES + [TARGET_COL]],
                     continuous_features=X_FEATURES, outcome_name=TARGET_COL)
    m = dice_ml.Model(model=mdl, backend='sklearn')
    exp = dice_ml.Dice(d, m, method='genetic')

    llm_ranges  = g['llm_raw_strict']
    hard_ranges = g['hardrule_ranges']
    final_ranges = g['final_ranges']

    conditions = {
        'PreGuardrail':  (None, None),
        'LLM_only':      (build_permitted(llm_ranges,  dref, X_FEATURES,
                                          only_these=set(llm_ranges)), llm_ranges),
        'HardRule_only': (build_permitted(hard_ranges, dref, X_FEATURES), hard_ranges),
        'LLMHard_soft':  (build_permitted(final_ranges, dref, X_FEATURES), final_ranges),
        'LLMHard_hard':  (build_permitted(final_ranges, dref, X_FEATURES), final_ranges),
    }

    for cond, (permitted, raw_ranges) in conditions.items():
        cfs = gen_cfs(exp, query, permitted)
        hard_clip = (cond == 'LLMHard_hard')

        # attribution: does the injected range itself permit a violation?
        if raw_ranges is not None:
            attribution.append({
                'CaseKey': case_key, 'Group': grp, 'Condition': cond,
                'range_permits_violation': range_permits_violation(orig, raw_ranges),
            })

        for i, cf_raw in enumerate(cfs, 1):
            # soft = as generated; hard = after deterministic hard-rule clip
            cf_soft = cf_raw
            cf_hard = clip_to_ranges(cf_raw, hard_ranges) if hard_clip else cf_raw

            cf_eval = cf_hard if hard_clip else cf_soft
            sc = score_all(cf_eval, orig)

            rec = {
                'CaseKey': case_key, 'Group': grp, 'Condition': cond, 'CF_idx': i,
                **sc,
                'range_respected': int(range_respected(cf_raw, permitted)) if permitted else 1,
            }
            records.append(rec)
    print(f"  [{case_key}] done")

per_cf = pd.DataFrame(records)
per_cf.to_csv('../results/tables/governance_07c_percf.csv', index=False, encoding='utf-8-sig')

attr = pd.DataFrame(attribution)
attr.to_csv('../results/tables/governance_07c_attribution.csv', index=False, encoding='utf-8-sig')

# %%
# ## 6. Summary: compliance by condition (external scoring)
summary = (per_cf.groupby('Condition')[['G1','G2','G3','G4']].mean() * 100).round(1)
order = ['PreGuardrail','LLM_only','HardRule_only','LLMHard_soft','LLMHard_hard']
summary = summary.reindex([c for c in order if c in summary.index])

# add optimiser-respect and range-permits-violation rates
resp = (per_cf.groupby('Condition')['range_respected'].mean() * 100).round(1)
summary['range_respected_%'] = resp.reindex(summary.index)
if len(attr):
    permit = (attr.groupby('Condition')['range_permits_violation'].mean() * 100).round(1)
    summary['range_permits_viol_%'] = permit.reindex(summary.index)

summary.to_csv('../results/tables/governance_07c_summary.csv', encoding='utf-8-sig')
print("\n=== 07c Summary (external scoring; compliance %) ===")
print(summary.to_string())

# %%
# ## 7. McNemar: the comparisons the reviewers asked for
def mcnemar_pair(a_cond, b_cond):
    a = per_cf[per_cf['Condition'] == a_cond].set_index(['CaseKey','CF_idx'])
    b = per_cf[per_cf['Condition'] == b_cond].set_index(['CaseKey','CF_idx'])
    idx = a.index.intersection(b.index)
    rows = []
    for dim in ['G1','G2','G3','G4']:
        av, bv = a.loc[idx, dim].values, b.loc[idx, dim].values
        n01 = int(((av==0)&(bv==1)).sum()); n10 = int(((av==1)&(bv==0)).sum())
        n11 = int(((av==1)&(bv==1)).sum()); n00 = int(((av==0)&(bv==0)).sum())
        try:
            p = mcnemar(np.array([[n11,n10],[n01,n00]]), exact=True).pvalue
        except Exception:
            p = np.nan
        rows.append({'Comparison': f'{a_cond}_vs_{b_cond}', 'Dim': dim,
                     'A_%': round(av.mean()*100,1), 'B_%': round(bv.mean()*100,1),
                     'Delta': round((bv.mean()-av.mean())*100,1),
                     'p_value': round(p,6) if not np.isnan(p) else np.nan,
                     'n_pairs': len(idx)})
    return rows

mc = []
for a, b in [('HardRule_only','LLMHard_soft'),   # Reviewer #5: does LLM add over hard rule?
             ('HardRule_only','LLMHard_hard'),
             ('LLM_only','LLMHard_soft'),         # does hard rule add over LLM?
             ('PreGuardrail','LLMHard_soft'),     # does the full guardrail help at all?
             ('LLMHard_soft','LLMHard_hard')]:    # measured value of the deterministic clip
    mc += mcnemar_pair(a, b)
mc_df = pd.DataFrame(mc)
mc_df.to_csv('../results/tables/governance_07c_mcnemar.csv', index=False, encoding='utf-8-sig')
print("\n=== 07c McNemar (external scoring) ===")
print(mc_df.to_string(index=False))

print("\nNOTE: external thresholds (EXT dict) are provisional and must be")
print("confirmed with the corresponding author / dietitian before submission.")

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.70it/s]


  [MiddleAged_Male_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.38it/s]


  [MiddleAged_Male_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.64it/s]


...

  [Older_Female_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.73it/s]


  [Older_Female_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.05it/s]

  [Older_Female_case3] done

=== 07c Summary (external scoring; compliance %) ===
                  G1     G2     G3    G4  range_respected_%  range_permits_viol_%
Condition                                                                        
PreGuardrail    93.8   35.4   62.5  62.5              100.0                   NaN
LLM_only        91.5   40.4   68.1  51.1              100.0                 100.0
HardRule_only   87.2   46.8   57.4  66.0              100.0                 100.0
LLMHard_soft    91.5   40.4   63.8  48.9              100.0                 100.0
LLMHard_hard   100.0  100.0  100.0  56.5              100.0                 100.0

=== 07c McNemar (external scoring) ===
                   Comparison Dim  A_%   B_%  Delta  p_value  n_pairs
HardRule_only_vs_LLMHard_soft  G1 87.2  91.5    4.3 0.625000       47
HardRule_only_vs_LLMHard_soft  G2 46.8  40.4   -6.4 0.663624       47
HardRule_only_vs_LLMHard_soft  G3 57.4  63.8    6.4 0.607239       47
HardRule_only_vs_LLMHard